[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Status Codes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `error_message` and
`decide`. Run it first.


In [1]:
import importlib
import sys
import urllib.request
from http import HTTPStatus
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()


def error_message(response):
    """What an error response says went wrong, read from its JSON body when it has one."""
    content_type = response.headers.get("Content-Type", "")
    if not content_type.startswith("application/json"):
        return f"{content_type} instead of JSON"
    body = response.json()
    return body.get("reason") or body.get("error")    # Open-Meteo says reason, the practice API error


def decide(response):
    """What to do about a response: an action, and the detail needed to take it."""
    code = response.status_code
    if code // 100 == 2:
        return "use", ("the body" if response.content else "no body")
    if code == HTTPStatus.NOT_FOUND:
        return "skip", error_message(response)
    if code in (HTTPStatus.UNAUTHORIZED, HTTPStatus.FORBIDDEN):
        return "stop", error_message(response)
    if code in (HTTPStatus.TOO_MANY_REQUESTS, HTTPStatus.SERVICE_UNAVAILABLE):
        return "wait", f"{int(response.headers.get('Retry-After', 60))} seconds"
    if code // 100 == 4:
        return "fix the request", error_message(response)
    if code // 100 == 5:
        return "try later", error_message(response)
    return "stop", f"no rule for status {code}"


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A status code, its reason, and its class.


In [2]:
response = requests.get(f"{BASE}/status/451", timeout=10)

print(response.status_code, response.reason, response.status_code // 100)


451 Unavailable For Legal Reasons 4


`451 Unavailable For Legal Reasons` is a `4xx`: the server will not serve the resource, for example
because a court has ordered it blocked.


**2.** Names and phrases from `HTTPStatus`.


In [3]:
for code in [409, 410]:
    status = HTTPStatus(code)
    print(status.name, "|", status.phrase, "| client error:", status.is_client_error)


CONFLICT | Conflict | client error: True
GONE | Gone | client error: True


`name` is what to write in code, as in `HTTPStatus.CONFLICT`, and `phrase` is what a server puts in
the status line.


**3.** Two waits, added together.


In [4]:
wait = 0
for code in [503, 429]:
    response = requests.get(f"{BASE}/status/{code}", timeout=10)
    wait += int(response.headers["Retry-After"])

print(wait, "seconds")


150 seconds


Each `Retry-After` value is a string, so `int` converts it before it is added.


**4.** A body only from a successful JSON response.


In [5]:
def json_or_none(response):
    """The parsed body of a successful JSON response, or None."""
    is_json = response.headers.get("Content-Type", "").startswith("application/json")
    if response.status_code // 100 == 2 and is_json:
        return response.json()
    return None


for path in ["/stations/oslo", "/status/204", "/stations/narvik", "/status/504"]:
    print(f"{path:<16}", json_or_none(requests.get(f"{BASE}{path}", timeout=10)))


/stations/oslo   {'id': 'oslo', 'name': 'Oslo', 'latitude': 59.91, 'longitude': 10.75}
/status/204      None
/stations/narvik None
/status/504      None


`/stations/narvik` has a JSON body, the practice API's error message, but its status is `404`, so
the function returns `None` instead of an error message that looks like data. `/status/204` passes
the status check and fails the `Content-Type` check, because a `204` has no body and so no
`Content-Type`.


**5.** The class of any status code.


In [6]:
def class_name(code):
    """The class of any status code, read from its first digit."""
    classes = {1: "informational", 2: "success", 3: "redirection", 4: "client error", 5: "server error"}
    return classes[code // 100]


for code in [204, 308, 451, 520, 599]:
    print(code, class_name(code))


204 success
308 redirection
451 client error
520 server error
599 server error


`code // 100` gives the class of any number a server sends, so `520` and `599` need nothing special.
A number outside 100 to 599 is not a status code, and raises `KeyError` here.


**6.** One more rule, ahead of `decide`.


In [7]:
def decide_gone(response):
    """decide, with one more rule: a 410 means the resource will not return."""
    if response.status_code == HTTPStatus.GONE:
        return "stop asking", error_message(response)
    return decide(response)


for code in [410, 404]:
    print(code, decide_gone(requests.get(f"{BASE}/status/{code}", timeout=10)))


410 ('stop asking', 'the resource has been removed, permanently')
404 ('skip', 'the resource does not exist')


The new rule runs before `decide`, so it takes priority for a `410`, and every other response keeps
the rules `decide` already has. Without it, a `410` would reach `decide`'s rule for any other `4xx`,
fix the request, when nothing about the request can be fixed.


---

&#8592; **Back to:** [Status Codes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/04-status-codes.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
